In [0]:
from pyspark.sql import functions as F

def normalize_columns(df):
    """Standardizes column names: lowercase, spaces to underscores."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

In [0]:
df_estimate = spark.table("workspace.bronze.estimate")
df_estimate = normalize_columns(df_estimate)

df_estimate = (
    df_estimate
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("version_no", F.col("version_no").cast("int"))
    .dropDuplicates()
)

df_estimate.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.estimate")
print(f"silver.estimate: {df_estimate.count()} rows written")